In [1]:
from pathlib import Path
import pandas as pd

In [2]:
YEAR = 2023
PROJECT_DIR = Path("../..").resolve()
print("Base Directory:", PROJECT_DIR)


Base Directory: /Users/nicholasbenelli/Workspace/repos/GitHub/-sports/-tennis/Tennis-Data-Pipeline


## Load Data

### Tennis Data UK

In [3]:
df_uk = pd.read_csv(f"../../data/clean/tennis-data-uk/atp/atp_singles_results_{YEAR}.csv")
print(list(df_uk.columns))

df_uk.head()

['ATP', 'Year', 'Location', 'Tournament', 'Date', 'Series', 'Court', 'Surface', 'Round', 'Best of', 'Winner', 'Loser', 'WRank', 'LRank', 'WPts', 'LPts', 'W1', 'L1', 'W2', 'L2', 'W3', 'L3', 'W4', 'L4', 'W5', 'L5', 'Wsets', 'Lsets', 'Comment', 'B365W', 'B365L', 'PSW', 'PSL', 'MaxW', 'MaxL', 'AvgW', 'AvgL']


,ATP,Year,Location,Tournament,Date,Series,Court,Surface,Round,Best of,...,Lsets,Comment,B365W,B365L,PSW,PSL,MaxW,MaxL,AvgW,AvgL
0,1,2023,Adelaide,Adelaide International 1,1/1/23,ATP250,Outdoor,Hard,1st Round,3,...,1.0,Completed,1.91,1.91,1.93,1.95,1.99,1.95,1.89,1.89
1,1,2023,Adelaide,Adelaide International 1,1/1/23,ATP250,Outdoor,Hard,1st Round,3,...,0.0,Retired,1.36,3.20,1.39,3.25,1.44,3.40,1.36,3.12
2,1,2023,Adelaide,Adelaide International 1,1/2/23,ATP250,Outdoor,Hard,1st Round,3,...,0.0,Completed,1.57,2.38,1.58,2.53,1.64,2.53,1.58,2.36
3,1,2023,Adelaide,Adelaide International 1,1/2/23,ATP250,Outdoor,Hard,1st Round,3,...,1.0,Completed,3.75,1.29,4.00,1.28,4.00,1.31,3.56,1.29
4,1,2023,Adelaide,Adelaide International 1,1/2/23,ATP250,Outdoor,Hard,1st Round,3,...,0.0,Completed,6.50,1.11,6.20,1.15,6.75,1.18,6.04,1.13


In [9]:
KEY_COLUMNS = ["ATP", "Year", "Location"]
INFO_COLS =  ["Tournament", "Series", "Court", "Surface", "Best of"]

Get Tournaments and their information

In [10]:
from tennis_data_pipeline.validatation.tennis_data_uk.tournaments import (
    find_uk_inconsistent_tournaments, 
    find_uk_reused_tournament_ids, 
)

In [12]:
inconsistent_tourneys = find_uk_inconsistent_tournaments(df_uk, key_columns=KEY_COLUMNS, info_cols=INFO_COLS)
inconsistent_tourneys

All tournament attributes are consistent.


(Empty DataFrame
 Columns: [Tournament, Series, Court, Surface, Best of]
 Index: [],
 Empty DataFrame
 Columns: [ATP, Year, Location, Tournament, Date, Series, Court, Surface, Round, Best of, Winner, Loser, WRank, LRank, WPts, LPts, W1, L1, W2, L2, W3, L3, W4, L4, W5, L5, Wsets, Lsets, Comment, B365W, B365L, PSW, PSL, MaxW, MaxL, AvgW, AvgL]
 Index: []
 
 [0 rows x 37 columns])

In [13]:
reused_tourney_id = find_uk_reused_tournament_ids(df=df_uk, id_col= "ATP", disambiguating_cols=["Location", "Tournament"])
reused_tourney_id

(     Location  Tournament
 ATP                      
 58          2           2,
     ATP  Year   Location                       Tournament      Date  Series  \
 0    58  2023  Stockholm                      Nordic Open  10/16/23  ATP250   
 1    58  2023  Stockholm                      Nordic Open  10/16/23  ATP250   
 2    58  2023  Stockholm                      Nordic Open  10/16/23  ATP250   
 3    58  2023  Stockholm                      Nordic Open  10/16/23  ATP250   
 4    58  2023  Stockholm                      Nordic Open  10/17/23  ATP250   
 5    58  2023  Stockholm                      Nordic Open  10/17/23  ATP250   
 6    58  2023  Stockholm                      Nordic Open  10/17/23  ATP250   
 7    58  2023  Stockholm                      Nordic Open  10/17/23  ATP250   
 8    58  2023  Stockholm                      Nordic Open  10/17/23  ATP250   
 9    58  2023  Stockholm                      Nordic Open  10/17/23  ATP250   
 10   58  2023  Stockholm             

In [18]:
import re


def slugify(value: str) -> str:
    value = value.strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")


def add_source_event_key(df):
    df = df.copy()

    # Date format drifts across seasons (e.g. "1/1/23" vs. "2023-01-01").
    df["Date"] = pd.to_datetime(df["Date"], format="mixed", errors="coerce")
    df["Year"] = df["Date"].dt.year

    df["source_event_key"] = (
        df["Year"].astype(str)
        + "_"
        + df["ATP"].astype(str)
        + "_"
        + df["Location"].map(slugify)
        + "_"
        + df["Tournament"].map(slugify)
    )

    return df


In [20]:
df_uk = add_source_event_key(df_uk)

In [ ]:
display(
    df_uk.loc[
        df_uk["ATP"].isin([38]),
        ["ATP", "Location", "Tournament", "Series", "Court", "Surface", "Best of", "Date"],
    ].drop_duplicates()
)


In [ ]:
uk_tourneys = df_uk['Tournament'].unique()
print(uk_tourneys)

In [ ]:
df_timl.loc[~df_timl['tourney_name'].str.contains("Davis Cup"), "tourney_name"].unique()

In [ ]:
source_df = df_uk.copy()
canonical_df = df_timl.loc[~df_timl['tourney_name'].str.contains("Davis Cup")].copy()


source_tournaments = (
    source_df[["Tournament"]]
    .drop_duplicates()
    .rename(columns={"Tournament": "source_tournament_name"})
)

canonical_tournaments = (
    canonical_df[["tourney_id", "tourney_name"]]
    .drop_duplicates()
    .rename(
        columns={
            "tourney_id": "canonical_tournament_id",
            "tourney_name": "canonical_tournament_name",
        }
    )
)

tournament_crosswalk = source_tournaments.merge(
    canonical_tournaments,
    left_on="source_tournament_name",
    right_on="canonical_tournament_name",
    how="left",
    validate="one_to_one",
)

tournament_crosswalk["match_method"] = "exact_name"
tournament_crosswalk["confidence"] = 1.0
tournament_crosswalk["review_flag"] = (
    tournament_crosswalk["canonical_tournament_id"].isna()
)

In [ ]:
tournament_crosswalk

In [ ]:
tournament_crosswalk["year"] = 2023
tournament_crosswalk["source"] = "tennis_data_uk"

In [ ]:
print(tournament_crosswalk)